In [1]:
import os
import sys
import logging
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
sys.path.append(project_root)
from config.conf import create_spark_session
from config.conf import config
import pyspark.sql.functions as F
from pyspark.sql.window import Window

In [2]:
spark = create_spark_session()

In [3]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

In [4]:
try:
    input_path = f"{config.root_path}{config.silver_zone}transfers"
    output_path = f"{config.root_path}{config.gold_zone}transfers_balance"
    
    logger.info(f"Lecture des données depuis : {input_path}")
    df_input = spark.read.option("header", "true").parquet(input_path)
    
    df_input = df_input.filter(F.col("transfer_fee") > 0)
    
    df = df_input.groupBy("to_club_name", "transfer_year") \
        .agg(
            F.count("to_club_name").alias("nb_purchases"),
            F.sum("transfer_fee").alias("total_purchases"),
            (F.sum("transfer_fee") / F.count("to_club_name")).cast("int").alias("mean_purchases"),
            F.sum("price_difference").alias("total_purchases_overprice"),
            (F.sum("price_difference") / F.count("to_club_name")).cast("int").alias("mean_purchases_overprice")
        ) \
        .withColumnRenamed("to_club_name", "club_name")
    
    sales = df_input.groupBy("from_club_name", "transfer_year") \
        .agg(
            F.count("from_club_name").alias("nb_sales"),
            F.sum("transfer_fee").alias("total_sales"),
            (F.sum("transfer_fee") / F.count("from_club_name")).cast("int").alias("mean_sales"),
            F.sum("price_difference").alias("total_sales_overprice"),
            (F.sum("price_difference") / F.count("from_club_name")).cast("int").alias("mean_sales_overprice")
        ) \
        .withColumnRenamed("from_club_name", "club_name")
    
    df = df.join(sales, on=["club_name", "transfer_year"], how="left") \
           .fillna(0) \
           .withColumn("balance", F.col("total_sales") - F.col("total_purchases"))
    
    logger.info(f"Écriture du Dataframe au format Parquet dans : {output_path}")
    df.write.mode("overwrite").parquet(output_path)
    logger.info("✅ Écriture terminée avec succès")
    
except Exception as e:
    logger.error(f"❌ Erreur lors du traitement de transfers : {e}", exc_info=True)

2025-05-19 09:39:38,528 - INFO - Lecture des données depuis : s3a://loicverdier/silver/football_transfermarkt/transfers
2025-05-19 09:39:45,814 - INFO - Écriture du Dataframe au format Parquet dans : s3a://loicverdier/gold/football_transfermarkt/transfers_balance
2025-05-19 09:39:55,314 - INFO - ✅ Écriture terminée avec succès                
